In [1]:
import pandas as pd

In [2]:
# Load datasets
grad_pct = pd.read_csv("../data/processed/grad_pct_per_DISTRICT.csv")
per_ecdis = pd.read_csv("../data/processed/PER_ECDIS_per_DISTRICT.csv")

In [ ]:
grad_pct

- Only keep data needed for analysis

Note: graduation rate is not recorded for cohorts of fewer than 5 students

In [3]:
# Only keep necessary rows
grad_pct = grad_pct.loc[(grad_pct.subgroup_code == 1) &  # Only keep 'All Students' numbers
                        (grad_pct.MEMBERSHIP_CODE == 9.0) & # Only keep 2021 Total Cohort - 4 Year Outcome
                        (grad_pct.GRAD_PCT != '-') # Drop all rows where grad pct is not recorded
                        ] 

# Drop all unnecessary columns
grad_pct = grad_pct.drop(['REPORT_SCHOOL_YEAR', 'AGGREGATION_TYPE', 'MEMBERSHIP_CODE', 'MEMBERSHIP_DESC', 'subgroup_code', 'SUBGROUP_NAME'], axis=1).reset_index(drop=True)

grad_pct

,AGGREGATION_CODE,AGGREGATION_NAME,ENROLL_CNT,GRAD_CNT,GRAD_PCT
0,240101040000,AVON CSD,73.0,69,95%
1,240901040000,MT MORRIS CSD,54.0,50,93%
2,241001060000,DANSVILLE CSD,122.0,100,82%
3,430901060000,GORHAM-MIDDLESEX CSD (MARCUS WHITMAN,93.0,81,87%
4,241101040000,DALTON-NUNDA CSD (KESHEQUA),52.0,47,90%
...,...,...,...,...,...
665,580912060000,EASTPORT-SOUTH MANOR CSD,285.0,257,90%
666,10623060000,NORTH COLONIE CSD,503.0,468,93%
667,212101040000,CENTRAL VALLEY CSD AT ILION-MOHAWK,166.0,133,80%
668,271201040000,OPPENHEIM-EPHRATAH-ST. JOHNSVILLE CS,53.0,47,89%


In [4]:
# Clean GRAD_PCT column
grad_pct['GRAD_PCT'] = '0.' + grad_pct['GRAD_PCT'].str[:-1]

# Convert metrics to appropriate types
grad_pct = grad_pct.astype({
    'AGGREGATION_CODE': str,
    'ENROLL_CNT': 'int32',
    'GRAD_CNT': 'int32',
    'GRAD_PCT': 'float32'
})

grad_pct

,AGGREGATION_CODE,AGGREGATION_NAME,ENROLL_CNT,GRAD_CNT,GRAD_PCT
0,240101040000,AVON CSD,73,69,0.95
1,240901040000,MT MORRIS CSD,54,50,0.93
2,241001060000,DANSVILLE CSD,122,100,0.82
3,430901060000,GORHAM-MIDDLESEX CSD (MARCUS WHITMAN,93,81,0.87
4,241101040000,DALTON-NUNDA CSD (KESHEQUA),52,47,0.90
...,...,...,...,...,...
665,580912060000,EASTPORT-SOUTH MANOR CSD,285,257,0.90
666,10623060000,NORTH COLONIE CSD,503,468,0.93
667,212101040000,CENTRAL VALLEY CSD AT ILION-MOHAWK,166,133,0.80
668,271201040000,OPPENHEIM-EPHRATAH-ST. JOHNSVILLE CS,53,47,0.89


Do the same for per_ecdis dataset

In [ ]:
per_ecdis

In [5]:
# Only keep year 2025, and drop year column
per_ecdis = per_ecdis.loc[per_ecdis.YEAR == 2025].reset_index(drop=True).drop('YEAR', axis=1)

# Keep percentage values between 0 and 1
per_ecdis['PER_ECDIS'] = per_ecdis['PER_ECDIS'] / 100

# Convert types
per_ecdis = per_ecdis.astype({
    'ENTITY_CD': 'str'
})

In [6]:
per_ecdis

,ENTITY_CD,ENTITY_NAME,PER_ECDIS
0,10100010000,ALBANY CITY SD,0.69
1,10201040000,BERNE-KNOX-WESTERLO CSD,0.36
2,10306060000,BETHLEHEM CSD,0.17
3,30200010000,BINGHAMTON CITY SD,0.80
4,30501040000,HARPURSVILLE CSD,0.64
...,...,...,...
712,671002040000,WYOMING CSD,0.35
713,671201060000,PERRY CSD,0.48
714,671501040000,WARSAW CSD,0.48
715,680601060000,PENN YAN CSD,0.55


In [10]:
master_dataset = grad_pct.merge(per_ecdis, left_on=['AGGREGATION_CODE', 'AGGREGATION_NAME'], right_on=['ENTITY_CD', 'ENTITY_NAME']).drop(['ENTITY_CD', 'ENTITY_NAME'],  axis=1).reset_index(drop=True)

master_dataset

,AGGREGATION_CODE,AGGREGATION_NAME,ENROLL_CNT,GRAD_CNT,GRAD_PCT,PER_ECDIS
0,240101040000,AVON CSD,73,69,0.95,0.34
1,240901040000,MT MORRIS CSD,54,50,0.93,0.64
2,241001060000,DANSVILLE CSD,122,100,0.82,0.55
3,430901060000,GORHAM-MIDDLESEX CSD (MARCUS WHITMAN,93,81,0.87,0.46
4,241101040000,DALTON-NUNDA CSD (KESHEQUA),52,47,0.90,0.47
...,...,...,...,...,...,...
663,580912060000,EASTPORT-SOUTH MANOR CSD,285,257,0.90,0.28
664,10623060000,NORTH COLONIE CSD,503,468,0.93,0.31
665,212101040000,CENTRAL VALLEY CSD AT ILION-MOHAWK,166,133,0.80,0.49
666,271201040000,OPPENHEIM-EPHRATAH-ST. JOHNSVILLE CS,53,47,0.89,0.64


In [11]:
master_dataset.to_csv('../data/processed/GRAD_PCT_vs_PER_ECDIS.csv', index=False)